In [87]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit, ParameterGrid
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    matthews_corrcoef,
)

In [88]:
data = pd.read_csv("eth_ml_dataset.csv")

In [ ]:
def _add_direction_extras(d: pd.DataFrame) -> pd.DataFrame:
    out = d.copy()
    eth = pd.to_numeric(out["ETH"], errors="coerce")
    btc = pd.to_numeric(out["BTC"], errors="coerce")
    er1 = pd.to_numeric(out["eth_ret_1d"], errors="coerce")
    br1 = pd.to_numeric(out["btc_ret_1d"], errors="coerce")
    if "eth_btc_price_ratio" not in out.columns:
        out["eth_btc_price_ratio"] = eth / (btc + 1e-9)
        out["eth_btc_ratio_ret_1d"] = out["eth_btc_price_ratio"].pct_change(1)
    if "eth_ret_1d_z7" not in out.columns:
        rm7e, rs7e = er1.rolling(7).mean(), er1.rolling(7).std()
        out["eth_ret_1d_z7"] = (er1 - rm7e) / (rs7e + 1e-9)
        rm7b, rs7b = br1.rolling(7).mean(), br1.rolling(7).std()
        out["btc_ret_1d_z7"] = (br1 - rm7b) / (rs7b + 1e-9)
        ev7 = pd.to_numeric(out["eth_vol_7d"], errors="coerce")
        out["eth_vol_7d_z7"] = (ev7 - ev7.rolling(7).mean()) / (ev7.rolling(7).std() + 1e-9)
        vx1 = pd.to_numeric(out["vix_ret_1d"], errors="coerce")
        out["vix_ret_1d_z7"] = (vx1 - vx1.rolling(7).mean()) / (vx1.rolling(7).std() + 1e-9)
        bes = pd.to_numeric(out["btc_eth_spread_1d"], errors="coerce")
        out["btc_eth_spread_1d_z7"] = (bes - bes.rolling(7).mean()) / (bes.rolling(7).std() + 1e-9)
    if "eth_ret_1d_lag1" not in out.columns and "sentiment_balance" in out.columns and "momentum_gap_7" in out.columns:
        st = pd.to_numeric(out["sentiment_balance"], errors="coerce")
        mg = pd.to_numeric(out["momentum_gap_7"], errors="coerce")
        for lag in (1, 2, 3):
            out[f"eth_ret_1d_lag{lag}"] = er1.shift(lag)
            out[f"btc_ret_1d_lag{lag}"] = br1.shift(lag)
            out[f"sentiment_balance_lag{lag}"] = st.shift(lag)
            out[f"momentum_gap_7_lag{lag}"] = mg.shift(lag)
    if "dow_sin" not in out.columns:
        dow = pd.to_datetime(out["date"]).dt.dayofweek.astype(float)
        out["dow_sin"] = np.sin(2 * np.pi * dow / 7.0)
        out["dow_cos"] = np.cos(2 * np.pi * dow / 7.0)
    if "eth_btc_ret_corr_30" not in out.columns:
        out["eth_btc_ret_corr_30"] = er1.rolling(30, min_periods=10).corr(br1)
    if "vix_x_eth_vol" not in out.columns:
        out["vix_x_eth_vol"] = pd.to_numeric(out["VIX"], errors="coerce") * pd.to_numeric(
            out["eth_vol_14d"], errors="coerce"
        )
    return out


df = data.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
df = _add_direction_extras(df)

df["y_eth_next_return"] = df["ETH"].shift(-1) / df["ETH"] - 1
df = df.iloc[:-1].copy()

MOVE_THRESHOLD = 0.003  
keep = df["y_eth_next_return"].abs() >= MOVE_THRESHOLD
df = df.loc[keep].reset_index(drop=True)

y = (df["y_eth_next_return"] > 0).astype(np.int8)

exclude_cols = {"date", "y_eth_next_return", "ETH"}
feature_cols = [
    c
    for c in df.columns
    if c not in exclude_cols and pd.api.types.is_numeric_dtype(df[c])
]
X = df[feature_cols]

TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.6, 0.2, 0.2
n = len(X)
n_test = int(n * TEST_FRAC)
n_val = int(n * VAL_FRAC)
n_train = n - n_val - n_test

X_train = X.iloc[:n_train]
y_train = y.iloc[:n_train]
X_val = X.iloc[n_train : n_train + n_val]
y_val = y.iloc[n_train : n_train + n_val]
X_test = X.iloc[n_train + n_val :]
y_test = y.iloc[n_train + n_val :]

_bad = []
for c in X.columns:
    s = X_train[c]
    if s.notna().sum() == 0:
        _bad.append(c)
    elif s.std(skipna=True) < 1e-12:
        _bad.append(c)
if _bad:
    X_train = X_train.drop(columns=_bad)
    X_val = X_val.drop(columns=_bad)
    X_test = X_test.drop(columns=_bad)
print("Dropped all-NaN / constant-on-train columns:", len(_bad))

X_train = X_train.to_numpy(dtype=np.float64)
X_val = X_val.to_numpy(dtype=np.float64)
X_test = X_test.to_numpy(dtype=np.float64)
y_train = y_train.to_numpy()
y_val = y_val.to_numpy()
y_test = y_test.to_numpy()

print("Rows after move filter:", n)
print("Train:", len(y_train), "| Val:", len(y_val), "| Test:", len(y_test))
print("Feature count:", X_train.shape[1])
print("Class balance (train) — class 0:", int((y_train == 0).sum()), "| class 1:", int((y_train == 1).sum()))

Dropped all-NaN / constant-on-train columns: 6
Rows after move filter: 1921
Train: 1153 | Val: 384 | Test: 384
Feature count: 163
Class balance (train) — class 0: 575 | class 1: 578


In [ ]:
_imputer_dt = SimpleImputer(strategy="median")
X_train_dt = _imputer_dt.fit_transform(X_train)
X_val_dt = _imputer_dt.transform(X_val)
for depth in [2, 4, 6, 10, 15, 20, 25, None]:
    clf = DecisionTreeClassifier(max_depth=depth, random_state=42)
    clf.fit(X_train_dt, y_train)
    pred_val = clf.predict(X_val_dt)
    acc = accuracy_score(y_val, pred_val)
    f1 = f1_score(y_val, pred_val, average="binary", zero_division=0)
    print(f"max_depth={depth!s:>4} | val accuracy={acc:.4f} | val F1={f1:.4f}")

max_depth=   2 | val accuracy=0.5026 | val F1=0.6583
max_depth=   4 | val accuracy=0.5104 | val F1=0.6557
max_depth=   6 | val accuracy=0.4948 | val F1=0.5126
max_depth=  10 | val accuracy=0.4740 | val F1=0.3484
max_depth=  15 | val accuracy=0.4896 | val F1=0.3836
max_depth=  20 | val accuracy=0.4870 | val F1=0.3901
max_depth=  25 | val accuracy=0.4818 | val F1=0.3642
max_depth=None | val accuracy=0.4818 | val F1=0.3642


In [ ]:

THRESHOLD_OBJECTIVE = "min_class_f1"  

CV_MACRO_WEIGHT = 0.55

TOP_K_RF_FEATURES = 100  

MODEL_SPECS = {
    "RandomForest": {
        "estimator": RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced_subsample"),
        "grid": {
            "clf__n_estimators": [200, 400],
            "clf__max_depth": [8, 12, None],
            "clf__min_samples_leaf": [1, 5, 10],
        },
        "needs_scaling": False,
    },
    "LogisticRegression": {
        "estimator": LogisticRegression(random_state=42, max_iter=6000, class_weight="balanced"),
        "grid": {
            "clf__C": [0.05, 0.2, 1.0, 5.0],
        },
        "needs_scaling": True,
    },
    "SGDClassifier": {
        "estimator": SGDClassifier(
            loss="log_loss",
            penalty="l2",
            random_state=42,
            max_iter=5000,
            tol=1e-3,
            class_weight="balanced",
        ),
        "grid": {
            "clf__alpha": [1e-5, 1e-4, 1e-3, 1e-2],
            "clf__learning_rate": ["optimal", "constant"],
            "clf__eta0": [0.01, 0.05],
        },
        "needs_scaling": True,
    },
    "KNN": {
        "estimator": KNeighborsClassifier(),
        "grid": {
            "clf__n_neighbors": [11, 21, 31, 41],
            "clf__weights": ["uniform", "distance"],
            "pca__n_components": [0.92, 0.95],
        },
        "needs_scaling": True,
        "use_pca": True,
    },
    "HistGBDT": {
        "estimator": HistGradientBoostingClassifier(
            random_state=42, early_stopping=False, class_weight="balanced"
        ),
        "grid": {
            "clf__max_iter": [150, 300],
            "clf__learning_rate": [0.03, 0.08],
            "clf__max_depth": [4, 8],
            "clf__min_samples_leaf": [10, 25],
        },
        "needs_scaling": False,
    },
}


def make_pipeline(base_estimator, needs_scaling, use_pca=False):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if needs_scaling:
        steps.append(("scaler", StandardScaler()))
    if use_pca:
        steps.append(("pca", PCA(random_state=42)))
    steps.append(("clf", base_estimator))
    return Pipeline(steps)


def evaluate_metrics(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "bal_acc": float(balanced_accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_pos": float(f1_score(y_true, y_pred, average="binary", zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
    }


def threshold_val_score(y_true, y_pred, objective: str) -> float:
    if objective == "macro_f1":
        return float(f1_score(y_true, y_pred, average="macro", zero_division=0))
    if objective == "balanced_accuracy":
        return float(balanced_accuracy_score(y_true, y_pred))
    if objective == "min_class_f1":
        f0, f1c = f1_score(y_true, y_pred, average=None, zero_division=0)
        return float(min(f0, f1c))
    if objective == "mcc":
        return float(matthews_corrcoef(y_true, y_pred))
    raise ValueError(objective)


def best_threshold_for_objective(probs_pos, y_true, objective: str):
    best_t, best_s = 0.5, -np.inf
    for t in np.linspace(0.05, 0.95, 91):
        pred = (probs_pos >= t).astype(int)
        s = threshold_val_score(y_true, pred, objective)
        if s > best_s:
            best_s, best_t = s, t
    return best_t, best_s


if TOP_K_RF_FEATURES is not None and X_train.shape[1] > TOP_K_RF_FEATURES:
    _rf_sel = RandomForestClassifier(
        n_estimators=400,
        max_depth=12,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced_subsample",
    )
    _im = SimpleImputer(strategy="median")
    Xtr_i = _im.fit_transform(X_train)
    _rf_sel.fit(Xtr_i, y_train)
    _idx = np.argsort(_rf_sel.feature_importances_)[::-1][:TOP_K_RF_FEATURES]
    X_train = X_train[:, _idx]
    X_val = X_val[:, _idx]
    X_test = X_test[:, _idx]
    print(f"Feature reduction: kept top {TOP_K_RF_FEATURES} by RF importance (train-only).")

tscv = TimeSeriesSplit(n_splits=4)
best_pipelines = {}
best_params_by_name = {}

for name, spec in MODEL_SPECS.items():
    best_score = -np.inf
    best_est = None
    best_params = None

    for params in ParameterGrid(spec["grid"]):
        pipe = make_pipeline(
            clone(spec["estimator"]),
            spec["needs_scaling"],
            spec.get("use_pca", False),
        )
        pipe.set_params(**params)

        fold_scores = []
        for tr_idx, cv_idx in tscv.split(X_train):
            X_tr_cv, X_cv = X_train[tr_idx], X_train[cv_idx]
            y_tr_cv, y_cv = y_train[tr_idx], y_train[cv_idx]
            pipe.fit(X_tr_cv, y_tr_cv)
            pred_cv = pipe.predict(X_cv)
            f1m = f1_score(y_cv, pred_cv, average="macro", zero_division=0)
            bacc = balanced_accuracy_score(y_cv, pred_cv)
            fold_scores.append(CV_MACRO_WEIGHT * f1m + (1.0 - CV_MACRO_WEIGHT) * bacc)

        mean_cv = float(np.mean(fold_scores))
        if mean_cv > best_score:
            best_score = mean_cv
            best_est = clone(pipe)
            best_params = params

    best_est.fit(X_train, y_train)
    best_pipelines[name] = best_est
    best_params_by_name[name] = best_params
    pred_val = best_est.predict(X_val)
    m = evaluate_metrics(y_val, pred_val)
    print(
        f"{name:18} best_cv_blend={best_score:.4f}  "
        f"val_f1_macro={m['f1_macro']:.4f}  val_bal_acc={m['bal_acc']:.4f}  params={best_params}"
    )

val_rank = []
for name, est in best_pipelines.items():
    pv = est.predict(X_val)
    mv = evaluate_metrics(y_val, pv)
    val_rank.append((name, mv))

val_rank = sorted(val_rank, key=lambda x: (x[1]["f1_macro"], x[1]["bal_acc"]), reverse=True)
selected_name = val_rank[0][0]

_knn_eps = 0.015
_nfeat = X_train.shape[1]
if selected_name == "KNN" and _nfeat >= 40 and len(val_rank) > 1:
    gap = val_rank[0][1]["f1_macro"] - val_rank[1][1]["f1_macro"]
    if gap < _knn_eps:
        _alt = val_rank[1][0]
        print(
            f"\nNote: KNN val-F1 lead vs {_alt} is only {gap:.4f} with {_nfeat} features; "
            f"using {_alt} for robustness."
        )
        selected_name = _alt

print(f"\nSelected model: {selected_name}")

spec_sel = MODEL_SPECS[selected_name]
cal_pipe = make_pipeline(
    clone(spec_sel["estimator"]),
    spec_sel["needs_scaling"],
    spec_sel.get("use_pca", False),
)
cal_pipe.set_params(**best_params_by_name[selected_name])
cal_pipe.fit(X_train, y_train)
probs_val = cal_pipe.predict_proba(X_val)[:, 1]

print("\n--- Drempels op validatie (zoek beste t per doel) ---")
for obj in ("macro_f1", "balanced_accuracy", "min_class_f1", "mcc"):
    t_opt, s_opt = best_threshold_for_objective(probs_val, y_val, obj)
    print(f"  {obj:22}  thr={t_opt:.4f}  val_score={s_opt:.4f}")

best_thr, best_val_score = best_threshold_for_objective(probs_val, y_val, THRESHOLD_OBJECTIVE)
print(
    f"\nGebruikt voor TEST: THRESHOLD_OBJECTIVE={THRESHOLD_OBJECTIVE!r}  "
    f"thr={best_thr:.4f}  (val score={best_val_score:.4f})"
)

X_trainval = np.vstack([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])
final_clf = make_pipeline(
    clone(spec_sel["estimator"]),
    spec_sel["needs_scaling"],
    spec_sel.get("use_pca", False),
)
final_clf.set_params(**best_params_by_name[selected_name])
final_clf.fit(X_trainval, y_trainval)
probs_test = final_clf.predict_proba(X_test)[:, 1]
y_pred_test = (probs_test >= best_thr).astype(int)

test_metrics = evaluate_metrics(y_test, y_pred_test)
print(f"\n=== TEST metrics ({THRESHOLD_OBJECTIVE}, thr from val) ===")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred_test, digits=4))
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, y_pred_test))

_thr_mf, _ = best_threshold_for_objective(probs_val, y_val, "macro_f1")
_y_mf = (probs_test >= _thr_mf).astype(int)
_m_mf = evaluate_metrics(y_test, _y_mf)
print(f"\n=== TEST ter referentie (macro_f1-drempel op val, thr={_thr_mf:.4f}) ===")
print(f"f1_macro: {_m_mf['f1_macro']:.4f}  bal_acc: {_m_mf['bal_acc']:.4f}  mcc: {_m_mf['mcc']:.4f}")

majority = int(np.bincount(y_trainval.astype(int)).argmax())
y_base = np.full_like(y_test, majority)
base_metrics = evaluate_metrics(y_test, y_base)
print("\n=== TEST baseline (always majority class from train+val) ===")
for k, v in base_metrics.items():
    print(f"{k}: {v:.4f}")
print(f"majority class predicted: {majority}")


Feature reduction: kept top 100 by RF importance (train-only).
RandomForest       best_cv_blend=0.4983  val_f1_macro=0.4969  val_bal_acc=0.5035  params={'clf__max_depth': None, 'clf__min_samples_leaf': 10, 'clf__n_estimators': 400}
LogisticRegression best_cv_blend=0.4775  val_f1_macro=0.4860  val_bal_acc=0.4933  params={'clf__C': 0.05}
SGDClassifier      best_cv_blend=0.4843  val_f1_macro=0.4625  val_bal_acc=0.4693  params={'clf__alpha': 0.01, 'clf__eta0': 0.01, 'clf__learning_rate': 'constant'}
KNN                best_cv_blend=0.5175  val_f1_macro=0.4505  val_bal_acc=0.4760  params={'clf__n_neighbors': 21, 'clf__weights': 'uniform', 'pca__n_components': 0.92}
HistGBDT           best_cv_blend=0.5142  val_f1_macro=0.4764  val_bal_acc=0.4861  params={'clf__learning_rate': 0.08, 'clf__max_depth': 4, 'clf__max_iter': 300, 'clf__min_samples_leaf': 25}

Selected model: RandomForest

--- Drempels op validatie (zoek beste t per doel) ---
  macro_f1                thr=0.4900  val_score=0.5182
 

In [ ]:

def _macro_f1_masked(y_true, y_pred, mask):
    yt = y_true[mask]
    yp = y_pred[mask]
    if len(yt) < 8:
        return float("nan")
    return float(f1_score(yt, yp, average="macro", zero_division=0))

_y_val = (probs_val >= best_thr).astype(int)
_y_test = (probs_test >= best_thr).astype(int)

margins = np.linspace(0.0, 0.45, 46)
best_m, best_vf1 = 0.0, -1.0
val_rows = []
for m in margins:
    mv = np.abs(probs_val - 0.5) >= m
    cov_v = float(mv.mean())
    f1v = _macro_f1_masked(y_val, _y_val, mv)
    val_rows.append((m, cov_v, f1v))
    if not np.isnan(f1v) and f1v > best_vf1:
        best_vf1, best_m = f1v, m

print("\n=== Selective: tune |P(up)-0.5| margin on validation (max subset macro-F1) ===")
print(f"Chosen margin = {best_m:.3f}  →  val subset macro-F1 = {best_vf1:.4f}")

mt = np.abs(probs_test - 0.5) >= best_m
print(
    f"TEST: coverage = {mt.mean():.1%} of days  |  macro-F1 on that subset = "
    f"{_macro_f1_masked(y_test, _y_test, mt):.4f}"
)

m60 = None
for m, cov_v, f1v in val_rows:
    if not np.isnan(f1v) and f1v >= 0.60:
        m60 = m
        break
if m60 is not None:
    msk = np.abs(probs_test - 0.5) >= m60
    print(
        f"\nMargin on VAL with subset macro-F1 >= 60%: >= {m60:.3f}  →  "
        f"TEST subset macro-F1 = {_macro_f1_masked(y_test, _y_test, msk):.4f}  "
        f"(coverage {msk.mean():.1%})"
    )
else:
    print(
        "\nNo validation margin reached 60% subset macro-F1. "
        "Try MOVE_THRESHOLD = 0.006–0.01 in the data cell (larger moves = easier labels)."
    )



=== Selective: tune |P(up)-0.5| margin on validation (max subset macro-F1) ===
Chosen margin = 0.060  →  val subset macro-F1 = 0.5411
TEST: coverage = 23.7% of days  |  macro-F1 on that subset = 0.5696

No validation margin reached 60% subset macro-F1. Try MOVE_THRESHOLD = 0.006–0.01 in the data cell (larger moves = easier labels).
